# Day 1 — Dataset & Problem Framing

## Business Problem
Customer churn is when a subscriber stops using a service. For a telecom company, acquiring a new customer costs **5–7× more** than retaining an existing one. If we can predict which customers are likely to leave, the retention team can intervene proactively with offers or support.

## ML Objective
> **Binary classification:** Given a customer's account details and usage patterns, predict whether they will churn (`Churn = 1`) or stay (`Churn = 0`).

## Dataset
- **Source:** Telco Customer Churn (Kaggle / IBM Sample Data)
- **Rows:** ~7,043 customers
- **Target column:** `Churn` (Yes / No)
- **Features:** Demographics, account info, services subscribed, billing details

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Load raw data ──────────────────────────────────────────────────────────────
df = pd.read_csv('../data/raw/Telco-Customer-Churn.csv')
print(f'Shape: {df.shape}')
print(f'Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}')

Shape: (7043, 21)
Rows: 7,043  |  Columns: 21


In [2]:
# ── First look ─────────────────────────────────────────────────────────────────
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# ── Column types and non-null counts ──────────────────────────────────────────
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [4]:
# ── Summary statistics for numeric columns ────────────────────────────────────
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [5]:
# ── Column breakdown: dtype + unique count + sample values ───────────────────
summary = pd.DataFrame({
    'dtype':   df.dtypes,
    'n_unique': df.nunique(),
    'sample':  [df[c].dropna().unique()[:3].tolist() for c in df.columns]
})
summary

,dtype,n_unique,sample
customerID,str,7043,"[7590-VHVEG, 5575-GNVDE, 3668-QPYBK]"
gender,str,2,"[Female, Male]"
SeniorCitizen,int64,2,"[0, 1]"
Partner,str,2,"[Yes, No]"
Dependents,str,2,"[No, Yes]"
tenure,int64,73,"[1, 34, 2]"
PhoneService,str,2,"[No, Yes]"
MultipleLines,str,3,"[No phone service, No, Yes]"
InternetService,str,3,"[DSL, Fiber optic, No]"
OnlineSecurity,str,3,"[No, Yes, No internet service]"


In [6]:
# ── Target column raw distribution ────────────────────────────────────────────
print('Churn value counts (raw):')
print(df['Churn'].value_counts())
print()
print('Churn proportions:')
print(df['Churn'].value_counts(normalize=True).round(3))

Churn value counts (raw):
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn proportions:
Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64


## Key observations
- `TotalCharges` is loaded as `object` (string) — needs fixing.
- `customerID` is an identifier — not a feature, will be dropped.
- ~26–27% of customers churned — **class imbalance** to handle in modeling.
- Many binary Yes/No columns for services (e.g. OnlineSecurity, TechSupport).